In [ ]:
# ==============================================================================
# CureSense — MedGemma Service (Google Colab)
# colab_medgemma.ipynb
#
# Runs: MedGemma 1.5-4B for CT/MRI and X-ray image analysis
# GPU:  T4 (16 GB) — dedicated entirely to MedGemma
# URL:  MEDGEMMA_SERVICE_URL in Backend/.env
#
# Colab secrets required (left panel → key icon → Add new secret):
#   HF_TOKEN          — HuggingFace token (accept MedGemma terms first at
#                       hf.co/google/medgemma-1.5-4b-it)
#   NGROK_AUTH_TOKEN  — ngrok auth token (ngrok.com → Your Authtoken)
#
# Run all cells top-to-bottom. Copy the printed MEDGEMMA_SERVICE_URL into
# Backend/.env and restart Express.
# ==============================================================================

In [ ]:
# ── Cell 1: Pull code from GitHub ────────────────────────────────────────────
import sys, os, subprocess

REPO_URL = 'https://github.com/moizaimran/curesense-project.git'
BRANCH   = 'hassan-branch'
REPO_DIR = '/content/curesense-project'

if not os.path.exists(REPO_DIR):
    print('Cloning repo ...')
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    print('Repo already cloned — pulling latest ...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

sys.path.insert(0, REPO_DIR)
print('Ready:', REPO_DIR)

In [ ]:
# ── Cell 2: Installs ──────────────────────────────────────────────────────────
import subprocess
subprocess.run(['pip', 'uninstall', '-y', 'torchaudio'], capture_output=True)
print('torchaudio removed')

# Pin to 4.53.0 — tested with MedGemma 1.5-4B; attn_implementation="eager" confirmed working.
!pip install -q "transformers==4.53.0"
!pip install -q accelerate pyngrok fastapi uvicorn pydicom httpx
!pip install -q pylibjpeg pylibjpeg-libjpeg

import transformers
print(f'Done — transformers {transformers.__version__}')

In [ ]:
# ── Cell 3: Colab secrets + HuggingFace login + ngrok auth ───────────────────
#
# Add secrets in Colab: left panel → key icon → Add new secret
#   HF_TOKEN          — HuggingFace read token
#   NGROK_AUTH_TOKEN  — ngrok auth token
#
from google.colab import userdata
from huggingface_hub import login
from pyngrok import ngrok

login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)
ngrok.set_auth_token(userdata.get('NGROK_AUTH_TOKEN'))

print('HuggingFace login OK')
print('ngrok ready')

In [ ]:
# ── Cell 4: Start MedGemma FastAPI on port 5002 ───────────────────────────────
#
# Colab T4 is the only GPU (GPU 0). _load_model() detects 1 GPU and
# automatically uses device_map='auto' — places everything on GPU 0.
#
import socket, threading, time, uvicorn
from api.medgemma_app import app as medgemma_app, _load_model

def _port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("localhost", port)) == 0

if _port_in_use(5002):
    print("[MedGemma] Already running on port 5002 — skipping restart")
else:
    threading.Thread(
        target=lambda: uvicorn.run(medgemma_app, host="0.0.0.0", port=5002, log_level="warning"),
        daemon=True,
    ).start()
    print("[MedGemma] FastAPI started on port 5002")

time.sleep(2)

# Pre-warm: downloads model weights (~8 GB) and prints VRAM stats
_load_model()

In [ ]:
# ── Cell 5: ngrok tunnel → MedGemma:5002 ─────────────────────────────────────
#
# Exposes the MedGemma FastAPI service at a public HTTPS URL.
# Copy the printed URL into Backend/.env as MEDGEMMA_SERVICE_URL.
#
from pyngrok import ngrok
import time

# Kill any existing ngrok process before starting fresh.
# Required when re-running this cell — the old process keeps the tunnel alive.
# Auth was already set in Cell 3; no need to fetch the secret again.
ngrok.kill()
time.sleep(2)

PUBLIC_URL = ngrok.connect(5002).public_url
print(f"[ngrok] Connected → {PUBLIC_URL} → MedGemma:5002")

print("\n" + "=" * 64)
print(f"  MEDGEMMA_SERVICE_URL = {PUBLIC_URL}")
print("=" * 64)
print("\nPaste into Backend/.env as MEDGEMMA_SERVICE_URL and restart Express.")
print("The Kaggle notebook provides AI_SERVICE_URL (Flask — PDF/interview).")

In [ ]:
# ── Cell 6: Show debug logs ───────────────────────────────────────────────────
# Run this cell AFTER triggering a scan from the app.
# It prints everything MedGemma logged during inference.
import os
LOG = "/tmp/medgemma_debug.log"
if os.path.exists(LOG):
    with open(LOG) as f:
        print(f.read())
else:
    print("No log yet — trigger a scan first, then re-run this cell.")

In [ ]:
# ── Cell 7: Multimodal diagnostic — run BEFORE triggering a real scan ─────────
# Tests the full vision+text pipeline with a blank test image.
# Tells us exactly what token the model would generate first with real inputs.
import sys, torch, numpy as np
from PIL import Image
sys.path.insert(0, '/content/curesense-project')
from api.medgemma_app import _load_model

model, processor = _load_model()

# Blank test image (black square — avoids any image-loading issues)
test_img = Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))

messages = [{"role": "user", "content": [
    {"type": "image", "image": test_img},
    {"type": "text", "text": "Describe this image briefly."},
]}]
text   = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
inputs = processor(text=text, images=test_img, return_tensors="pt").to(model.device)
inputs.pop("token_type_ids", None)
if "pixel_values" in inputs:
    inputs["pixel_values"] = inputs["pixel_values"].to(dtype=torch.float16)

prompt_len = inputs["input_ids"].shape[1]
print(f"prompt_len={prompt_len}  pixel_values dtype={inputs['pixel_values'].dtype}")
print(f"image token IDs in input (unique): {inputs['input_ids'].unique().tolist()[:10]}")

with torch.inference_mode():
    with torch.autocast("cuda", dtype=torch.float16):
        # Single forward pass — check if logits are valid
        out = model(**inputs)

print(f"\nlogits NaN?  {torch.isnan(out.logits).any().item()}")
print(f"logits last pos first 5: {out.logits[0, -1, :5].tolist()}")
print(f"argmax (would-be first token): {out.logits[0, -1].argmax().item()}")
print(f"top-5 token IDs: {out.logits[0, -1].topk(5).indices.tolist()}")

# Also generate 5 tokens to see what actually comes out
with torch.inference_mode():
    with torch.autocast("cuda", dtype=torch.float16):
        gen_ids = model.generate(**inputs, max_new_tokens=10, do_sample=False)

print(f"\ngenerated token IDs (first 10): {gen_ids[0][prompt_len:].tolist()}")
print(f"decoded: {processor.decode(gen_ids[0][prompt_len:], skip_special_tokens=True)!r}")